SEMANTIC CHUNKING

here docs are split into sentences and para, sentences are tested for similarity using cosine similarity test or some other test, similar sentences are merged based on a threshold. thus creating rich chunks which carry more context and info
after this embedding is done

In [7]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [6]:
# initializing the model
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
# sample text 
text = """
LangChain is a framework for building applications with LLMs.
LangChain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory and retrievers.
The Effiel Tower is located in Paris.
France is a popular tourist destination.
"""

# splitting into sentences
sentences = [s.strip() for s in text.split("\n") if s.strip()]

# embedding each sentence
embeddings = model.encode(sentences)

# initializing params
threshold = 0.7
chunks = []
current_chunk = [sentences[0]]

# semantic grouping based on threshold
for i in range(1,len(sentences)):
    sim = cosine_similarity(
        [embeddings[i-1]],
        [embeddings[i]]
    )[0][0]
    if sim>=threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk=[sentences[i]]

chunks.append(" ".join(current_chunk))

# testing output
for idx,chunk in enumerate(chunks):
    print(f"\n Chunk {idx+1}:\n{chunk}")

<class 'str'>

 Chunk 1:
LangChain is a framework for building applications with LLMs. LangChain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.

 Chunk 2:
You can create chains, agents, memory and retrievers.

 Chunk 3:
The Effiel Tower is located in Paris.

 Chunk 4:
France is a popular tourist destination.


RAG PIPELINE USING SEMANTIC CHUNKING

In [26]:
# libraries required
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnablePassthrough
from typing import List
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


CUSTOM SEMANTIC CHUNKER WITH THRESHOLD

In [14]:
class ThresholdSemanticChunker:
    def __init__(self,model_name="all-MiniLM-L6-v2",threshold=0.7):
        self.model = SentenceTransformer(model_name)
        self.threshold = threshold
    
    def split(self,text: str):
        sentences = [s.strip() for s in text.split('.') if s.strip()]
        embeddings = self.model.encode(sentences)
        chunks=[]
        current_chunk = [sentences[0]]

        for i in range(1,len(sentences)):
            sim = cosine_similarity([embeddings[i-1]],[embeddings[i]])[0][0]
            if sim >= self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(". ".join(current_chunk) + ".")
                current_chunk = [sentences[i]]
        
        chunks.append(". ".join(current_chunk) + ".")
        return chunks
    
    def split_documents(self,docs):
        result = []
        for doc in docs:
            for chunk in self.split(doc.page_content):
                result.append(
                    Document(
                        page_content=chunk,
                        metadata=doc.metadata
                    )
                )
        return result

testing out semantic chunker

In [15]:
sample_text = """
LangChain is a framework for building applications with LLMs.
LangChain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory and retrievers.
The Effiel Tower is located in Paris.
France is a popular tourist destination.
"""
doc = Document(page_content=sample_text)
doc


Document(metadata={}, page_content='\nLangChain is a framework for building applications with LLMs.\nLangChain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.\nYou can create chains, agents, memory and retrievers.\nThe Effiel Tower is located in Paris.\nFrance is a popular tourist destination.\n')

In [17]:
chunker = ThresholdSemanticChunker(threshold=0.7)
final_chunks = chunker.split_documents([doc])
final_chunks

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[Document(metadata={}, page_content='LangChain is a framework for building applications with LLMs. LangChain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.'),
 Document(metadata={}, page_content='You can create chains, agents, memory and retrievers.'),
 Document(metadata={}, page_content='The Effiel Tower is located in Paris.'),
 Document(metadata={}, page_content='France is a popular tourist destination.')]

Vectorstore and retriever setup

In [22]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding = HuggingFaceEmbeddings(
    model_name='all-MiniLM-L6-v2'
)
vectorstore = FAISS.from_documents(
    documents = final_chunks,
    embedding = embedding
)
retriever = vectorstore.as_retriever()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Prompt template setup

In [24]:
simple_prompt = ChatPromptTemplate.from_template("""Answer the question based only
on the following context:
Context: {context}
Question: {question}
Answer:""")

Initializing LLM model

In [25]:
llm = init_chat_model(
    model="groq:llama-3.1-8b-instant"
)

In [27]:
# format doc function for llm
def format_docs(docs: List[Document]) -> str:
    formatted=[]
    for i,doc in enumerate(docs):
        source = doc.metadata.get('source','unkown')
        formatted.append(f"Document {i+1} (Source: {source})\n{doc.page_content}")
    return "\n\n".join(formatted)

LCEL RAG Pipeline

In [29]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | simple_prompt
    | llm
    | StrOutputParser()
)
rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001DFA57DD400>, search_kwargs={})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only\non the following context:\nContext: {context}\nQuestion: {question}\nAnswer:'), additional_kwargs={})])
| ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000

testing via query

In [30]:
query = "What is Langchain used for?"
answer = rag_chain.invoke(query)
print(answer)

Based on the provided context, LangChain is a framework for building applications with Large Language Models (LLMs).


Semantic Chunker with Langchain (Inbuilt)

In [31]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.document_loaders import TextLoader

In [33]:
# loading the docs
loader=TextLoader("temp.txt")
docs = loader.load()
docs

[Document(metadata={'source': 'temp.txt'}, page_content='LangChain is a framework for building applications with LLMs.\nLangChain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.\nYou can create chains, agents, memory and retrievers.\nThe Effiel Tower is located in Paris.\nFrance is a popular tourist destination.')]

In [34]:
# creating Semantic Chunker
chunker2 = SemanticChunker(embedding)
chunks2 = chunker2.split_documents(docs)

for i,chunk in enumerate(chunks):
    print(f"\nChunk {i+1}:\n {chunk}")


Chunk 1:
 LangChain is a framework for building applications with LLMs. LangChain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.

Chunk 2:
 You can create chains, agents, memory and retrievers.

Chunk 3:
 The Effiel Tower is located in Paris.

Chunk 4:
 France is a popular tourist destination.
